# Dashboard Iris (Dash)

Info panel + graf, filtrovacie slidery, tabuľka. Presne 3 callbacky (info panel, graf, tabuľka), všetky reagujú na slidery.

In [ ]:
# !pip install dash pandas plotly
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, dash_table, Input, Output

## Načítanie dát

In [ ]:
# Načítanie datasetu Iris z CSV súboru
url = "https://raw.githubusercontent.com/uiuc-cse/data-fa14/gh-pages/data/iris.csv"
df = pd.read_csv(url)
df.head()

## Filter a slidery

In [ ]:
NUMERIC_COLS = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
RANGES = {col: (float(df[col].min()), float(df[col].max())) for col in NUMERIC_COLS}


def filter_df(*ranges):
    # Riadky vo zvolených rozsahoch (vrátane hraníc)
    filtered = df
    for col, (lo, hi) in zip(NUMERIC_COLS, ranges):
        filtered = filtered[filtered[col].between(lo, hi)]
    return filtered


def make_slider(col):
    # Popis + RangeSlider pre jeden stĺpec
    lo, hi = RANGES[col]
    return html.Div([
        html.Label(f"Rozsah {col}"),
        dcc.RangeSlider(id=f"slider-{col}", min=lo, max=hi, step=0.1, value=[lo, hi],
                        tooltip={"placement": "bottom", "always_visible": True}),
    ], style={"marginBottom": "25px"})


# Vstupy spoločné pre všetky 3 callbacky
SLIDER_INPUTS = [Input(f"slider-{col}", "value") for col in NUMERIC_COLS]

## Rozloženie

In [ ]:
BOX = {"border": "2px solid black", "padding": "10px", "margin": "5px"}

app = Dash(__name__)

app.layout = html.Div([
    # Info panel (30 %) + graf (70 %)
    html.Div([
        html.Div(id="info-panel", style={**BOX, "width": "30%"}),
        html.Div(dcc.Graph(id="graph"), style={**BOX, "width": "70%", "minWidth": 0}),
    ], style={"display": "flex"}),
    # Slidery
    html.Div([make_slider(col) for col in NUMERIC_COLS], style=BOX),
    # Tabuľka
    html.Div(dash_table.DataTable(
        id="table",
        columns=[{"name": col, "id": col} for col in df.columns],
        page_size=10,
        sort_action="native",
        style_table={"overflowX": "auto"},
        style_cell={"textAlign": "left", "padding": "5px"},
    ), style=BOX),
], style=BOX)

## Callbacky

In [ ]:
# 1. slidery -> info panel
@app.callback(Output("info-panel", "children"), SLIDER_INPUTS)
def update_info(*ranges):
    filtered = filter_df(*ranges)
    counts = filtered["species"].value_counts()
    return [
        html.H4("Info panel"),
        html.P(f"Počet filtrovaných záznamov: {len(filtered)}"),
        html.P(f"Počet všetkých záznamov: {len(df)}"),
        html.P("Počty podľa druhu:"),
        html.Ul([html.Li(f"{species}: {count}") for species, count in counts.items()]),
    ]

In [ ]:
# 2. slidery -> graf
@app.callback(Output("graph", "figure"), SLIDER_INPUTS)
def update_graph(*ranges):
    fig = px.scatter(filter_df(*ranges), x="sepal_length", y="petal_length", color="species",
                     title="Filtrované dáta")
    # Pevné osi, aby graf pri filtrovaní neposkakoval
    fig.update_xaxes(range=[df["sepal_length"].min() - 0.5, df["sepal_length"].max() + 0.5])
    fig.update_yaxes(range=[df["petal_length"].min() - 0.5, df["petal_length"].max() + 0.5])
    return fig

In [ ]:
# 3. slidery -> tabuľka
@app.callback(Output("table", "data"), SLIDER_INPUTS)
def update_table(*ranges):
    return filter_df(*ranges).to_dict("records")

## Spustenie

In [ ]:
app.run(debug=True, jupyter_mode="inline")